In [71]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np
import re

import sys
from collections import defaultdict

In [9]:
sys.argv = [
    'notebook',  
    '--modelname', 'llama3.2-vision:90b',
    '--data', 'gully',
    '--data_path','/root/home/WACV-2025-Workshop-ViGIR/results/proposed/valid_llama3_90b.json',
    '--subset', 'train',
    '--results_dir', '/root/home/WACV-2025-Workshop-ViGIR/results/',
    '--timeout', '20',
    '--model_unloading'
]

In [10]:
parser = argparse.ArgumentParser(description="A script to evaluate V-LLMs on different image classification datasets")

parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 samples it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [11]:
# Load test set:
file_path = os.path.join(args.data_path)
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  310
Number of Annotated GT Images (List):  310


In [12]:
data

{'356': [[{'label': '4', 'labelers': ['Pankaj', 'Parth']},
   'Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!',
   'No.'],
  [{'label': '4', 'labelers': ['Pankaj', 'Parth']},
   'Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!',
   'No.'],
  [{'label': '4', 'labelers': ['Pankaj', 'Parth']},
   'Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!',
   'No.'],
  [{'label': '4', 'labelers': ['Pankaj', 'Parth']},
   'Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!',
   'No.'],
  [

In [33]:
data_reformated = {}

for key, item in data.items():
    q_and_a = []
    for i in range(len(item)):
        tmp = item[i][1:]   
        q_and_a.append(tmp)

    data_reformated[key] = q_and_a
    

In [34]:
data_reformated

{'356': [['Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!',
   'No.'],
  ['Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes

In [54]:
data_prompt = {}
for key, item in data_reformated.items():
    prompt = "\n".join([f"Q: {qa[0]}\nA: {qa[1]}" for qa in item])
    data_prompt[key] = [prompt]

In [55]:
data_prompt

{'356': ['Q: Given these six images of the exact same area and collected over a period of 10 years, do you see a low point in the terrain? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, do narrow, winding paths or channels appear? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there winding paths that become intermittent recurrent? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there any linear depressions or ruts which appear more pronounced along natural drainage lines or slopes? Answer with yes or no only!\nA: No.\nQ: Given these six images of the exact same area and collected over a period of 10 years, are there narrow and shallow channels which appear intermittently deeper or more indented into the soil? Answer with yes or no only!\nA: N

In [56]:
model_name = args.modelname
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [59]:
count = 0
saving_response=copy.deepcopy(data_prompt)

for key, info in tqdm(data_prompt.items()):

    #print(key)
    #print(info[0])
    #sys.exit()
    question = "Based on the following questions and their answers, determine if there is evidence of an ephemeral gully in the observed area. Carefully analyze all the questions and the responses to assess.\n\n"
    question += info[0]
    question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."
    
    #print(question)
    #sys.exit()
        
    count+=1
    #for question in questions:
    response = ollama.generate(model=model_name, 
                               prompt=question, 
                               #images=info, 
                               options=options)
    saving_response[key].append(response['response'])
    print(response['response'])

    #sys.exit()
        

  0%|▎                                                                                                     | 1/310 [00:03<20:07,  3.91s/it]

Yes. 

Although most questions received negative answers, two key indicators were present: areas where soil appears disturbed or vegetation is removed (Q6) and a specific path lacks vegetation, suggesting an evolving or emerging channel (Q7). These signs are consistent with the characteristics of ephemeral gullies, which often form through erosion in areas with disturbed soil or reduced vegetation cover.


  1%|▋                                                                                                     | 2/310 [00:11<31:28,  6.13s/it]

Yes. 

Although most questions resulted in "no" answers, which might initially suggest the absence of an ephemeral gully, certain affirmative responses hint at characteristics that could be indicative of such a feature. Specifically, the presence of areas where soil appears disturbed or vegetation is removed (Q6), and the indication of a specific path lacking vegetation suggesting an evolving or emerging channel (Q7), along with varying exposure of lighter or darker colored soil (Q11), collectively provide evidence suggestive of ephemeral gully formation. These signs are consistent with the early stages of gully development, where initial disturbances in soil and vegetation can precede more pronounced morphological changes like channels and rills.


  1%|█▎                                                                                                    | 4/310 [00:12<12:08,  2.38s/it]

...

No.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████▋| 309/310 [20:01<00:01,  1.72s/it]

No.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 310/310 [20:14<00:00,  3.92s/it]

Yes. 

Although most questions resulted in "no" answers, which might initially suggest the absence of an ephemeral gully, several key indicators were present:

1. **Soil Disturbance and Vegetation Removal**: The presence of areas where soil appears disturbed or vegetation is removed (Q6) can be indicative of water flow or erosion, common in ephemeral gullies.

2. **Lack of Vegetation Suggesting an Emerging Channel**: A specific path lacking vegetation (Q7), suggesting an evolving or emerging channel, is a strong indicator of potential ephemeral gully formation.

3. **Varying Soil Texture and Color Exposure**: The varying types and levels of coarseness in the texture of the soil (Q8) and the varying exposure of lighter or darker colored soil (Q11) can indicate erosion and deposition processes typical of ephemeral gullies.

These indicators collectively suggest that despite the absence of more overt signs like linear depressions, winding paths, or sediment accumulations, there is indeed 

In [69]:
def extract_first_yes_no(text):
    # Use regular expressions to find "Yes" or "No" (case-insensitive)
    match = re.search(r'\b(Yes|No)\b', text, re.IGNORECASE)
    if match:
        # Return the first match in its original case
        return match.group(0)
    return None

In [72]:
saving_response_reformat = {}

for key, item in saving_response.items():
    print(key)
    print(item[-1])
    text = extract_first_yes_no(item[-1])
    print('\n\n\n', text)
    sys.exit()

356
Yes. 

Although most questions received negative answers, two key indicators were present: areas where soil appears disturbed or vegetation is removed (Q6) and a specific path lacks vegetation, suggesting an evolving or emerging channel (Q7). These signs are consistent with the characteristics of ephemeral gullies, which often form through erosion in areas with disturbed soil or reduced vegetation cover.



 Yes


SystemExit: 